In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import zlib
import jinja2
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

TARGETS = ["Revenue", "COGS"]

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/datathon-2026-round-1")
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "notebooks").exists():
    LOCAL_DATA_DIR = REPO_ROOT 
else:
    LOCAL_DATA_DIR = REPO_ROOT.parent

if KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR
    WORK_DIR = Path("/kaggle/working")
else:
    DATA_DIR = LOCAL_DATA_DIR
    WORK_DIR = REPO_ROOT

print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)


In [ ]:
def read_csv_optional(name, parse_dates=None, **kwargs):
    path = DATA_DIR / name
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=parse_dates, **kwargs)

sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", parse_dates=["Date"]).reset_index(drop=True)

orders = read_csv_optional("orders.csv", parse_dates=["order_date"])
order_items = read_csv_optional("order_items.csv", low_memory=False)
web = read_csv_optional("web_traffic.csv", parse_dates=["date"])
promotions = read_csv_optional("promotions.csv", parse_dates=["start_date", "end_date"])

print("sales:", sales.shape, sales["Date"].min().date(), "->", sales["Date"].max().date())
print("sample:", sample.shape, sample["Date"].min().date(), "->", sample["Date"].max().date())
print("optional tables:")
for name, df in {"orders": orders, "order_items": order_items, "web_traffic": web, "promotions": promotions}.items():
    print(f"  {name:12s}", None if df is None else df.shape)

assert list(sample.columns) == ["Date", "Revenue", "COGS"]
assert sales["Date"].is_monotonic_increasing
assert sample["Date"].is_monotonic_increasing


In [ ]:
def make_seasonal_profile_baseline(train_df, future_dates, level_source_year=2022):
    hist = train_df.copy()
    hist["year"] = hist["Date"].dt.year
    hist["month"] = hist["Date"].dt.month
    hist["day"] = hist["Date"].dt.day
    hist["rev_norm"] = hist["Revenue"] / hist.groupby("year")["Revenue"].transform("mean")
    hist["cogs_ratio"] = hist["COGS"] / hist["Revenue"].replace(0, np.nan)

    profile = hist.groupby(["month", "day"], as_index=False)["rev_norm"].median()
    ratio_source = hist[hist["year"] >= 2019].copy()
    if ratio_source.empty:
        ratio_source = hist.copy()
    recent_ratio = ratio_source.groupby("month")["cogs_ratio"].median()
    fallback_ratio = float(ratio_source["cogs_ratio"].median())

    future = pd.DataFrame({"Date": pd.to_datetime(future_dates)})
    future["month"] = future["Date"].dt.month
    future["day"] = future["Date"].dt.day
    future = future.merge(profile, on=["month", "day"], how="left")
    future["rev_norm"] = future["rev_norm"].fillna(1.0)

    base_rev_mean = hist.loc[hist["year"] == level_source_year, "Revenue"].mean()
    rev = future["rev_norm"].to_numpy(float) * float(base_rev_mean)
    cogs = rev * future["month"].map(recent_ratio).fillna(fallback_ratio).to_numpy(float)

    out = pd.DataFrame({"Date": future["Date"], "Revenue": rev, "COGS": cogs})
    return out

baseline_profile = make_seasonal_profile_baseline(sales, sample["Date"], level_source_year=2022)

print("Seasonal profile baseline level:")
print("  Revenue mean:", f"{baseline_profile['Revenue'].mean():,.0f}")
print("  COGS mean:   ", f"{baseline_profile['COGS'].mean():,.0f}")
display(baseline_profile.head())


In [ ]:
# Tính target statistics từ training data
train_mean_revenue = sales["Revenue"].mean()
train_mean_cogs = sales["COGS"].mean()

# Tính mean của baseline prediction
pred_mean_revenue = baseline_profile["Revenue"].mean()
pred_mean_cogs = baseline_profile["COGS"].mean()

print("Training data statistics:")
print(f"  Revenue mean: {train_mean_revenue:,.0f}")
print(f"  COGS mean:    {train_mean_cogs:,.0f}")
print("\nBaseline prediction statistics:")
print(f"  Revenue mean: {pred_mean_revenue:,.0f}")
print(f"  COGS mean:    {pred_mean_cogs:,.0f}")

# Target Fit: Điều chỉnh offset
revenue_offset = train_mean_revenue - pred_mean_revenue
cogs_offset = train_mean_cogs - pred_mean_cogs

print(f"\nOffset cần điều chỉnh:")
print(f"  Revenue offset: {revenue_offset:,.0f}")
print(f"  COGS offset:    {cogs_offset:,.0f}")

# Áp dụng Target Fit
submission_targetfit = baseline_profile.copy()
submission_targetfit["Revenue"] = submission_targetfit["Revenue"] + revenue_offset
submission_targetfit["COGS"] = submission_targetfit["COGS"] + cogs_offset
submission_targetfit[TARGETS] = submission_targetfit[TARGETS].clip(lower=0)  # Không âm

print("\nTarget Fit submission mean:")
print(f"  Revenue mean: {submission_targetfit['Revenue'].mean():,.0f}")
print(f"  COGS mean:    {submission_targetfit['COGS'].mean():,.0f}")

# So sánh 2 versions
print("\n=== So sánh Baseline vs Target Fit ===")
comparison = pd.DataFrame({
    "Metric": ["Revenue mean", "Revenue std", "COGS mean", "COGS std"],
    "Baseline": [
        baseline_profile["Revenue"].mean(),
        baseline_profile["Revenue"].std(),
        baseline_profile["COGS"].mean(),
        baseline_profile["COGS"].std(),
    ],
    "TargetFit": [
        submission_targetfit["Revenue"].mean(),
        submission_targetfit["Revenue"].std(),
        submission_targetfit["COGS"].mean(),
        submission_targetfit["COGS"].std(),
    ],
})
display(comparison.round(2))

In [ ]:
## 7. Multi-Target Scaling - Fit each target separately

# Multi-target scaling: Fit Revenue and COGS independently with both scale + offset
# Formula: pred_adjusted = pred * scale_factor + offset

print("=" * 60)
print("MULTI-TARGET SCALING - Independent fitting per target")
print("=" * 60)

multi_target_params = {}

for target in TARGETS:
    train_values = sales[target].values.astype(float)
    pred_values = baseline_profile[target].values.astype(float)
    
    # Calculate statistics
    train_mean = train_values.mean()
    train_std = train_values.std()
    pred_mean = pred_values.mean()
    pred_std = pred_values.std()
    
    # Method 1: Scale by std ratio + mean offset (handles both variance and bias)
    scale_factor = train_std / pred_std if pred_std > 0 else 1.0
    offset = train_mean - (pred_mean * scale_factor)
    
    multi_target_params[target] = {
        "train_mean": train_mean,
        "train_std": train_std,
        "pred_mean": pred_mean,
        "pred_std": pred_std,
        "scale_factor": scale_factor,
        "offset": offset,
    }
    
    print(f"\n{target}:")
    print(f"  Training:   mean={train_mean:,.0f}, std={train_std:,.0f}")
    print(f"  Prediction: mean={pred_mean:,.0f}, std={pred_std:,.0f}")
    print(f"  Fit params: scale={scale_factor:.4f}, offset={offset:,.0f}")

# Apply Multi-Target Scaling
submission_multitarget = baseline_profile.copy()

for target in TARGETS:
    params = multi_target_params[target]
    submission_multitarget[target] = (
        baseline_profile[target] * params["scale_factor"] + params["offset"]
    )

submission_multitarget[TARGETS] = submission_multitarget[TARGETS].clip(lower=0)

print("\n" + "=" * 60)
print("RESULTS COMPARISON")
print("=" * 60)

comparison_multitarget = pd.DataFrame({
    "Target": [],
    "Baseline Mean": [],
    "Baseline Std": [],
    "TargetFit Mean": [],
    "TargetFit Std": [],
    "MultiTarget Mean": [],
    "MultiTarget Std": [],
})

rows_comp = []
for target in TARGETS:
    rows_comp.append({
        "Target": target,
        "Baseline Mean": baseline_profile[target].mean(),
        "Baseline Std": baseline_profile[target].std(),
        "TargetFit Mean": submission_targetfit[target].mean(),
        "TargetFit Std": submission_targetfit[target].std(),
        "MultiTarget Mean": submission_multitarget[target].mean(),
        "MultiTarget Std": submission_multitarget[target].std(),
    })

comparison_multitarget = pd.DataFrame(rows_comp)
display(comparison_multitarget.round(2))

In [ ]:
# Save Multi-Target Scaling submission
multitarget_path = WORK_DIR / "multitarget_submission.csv"
submission_multitarget.to_csv(multitarget_path, index=False)

print(f"\nMulti-Target submission saved to: {multitarget_path}")
print(f"Shape: {submission_multitarget.shape}")
print("\nFirst 10 rows:")
display(submission_multitarget.head(10))

# Validation: Check for negative values
if (submission_multitarget[TARGETS] < 0).any().any():
    print("⚠️ WARNING: Found negative values after clipping!")
else:
    print("✓ All values are non-negative")

# Summary statistics
print("\n" + "=" * 60)
print("MEAN ABSOLUTE ERROR vs Training Distribution")
print("=" * 60)

mae_comparison = []
for target in TARGETS:
    mae_baseline = np.abs(baseline_profile[target].mean() - sales[target].mean())
    mae_targetfit = np.abs(submission_targetfit[target].mean() - sales[target].mean())
    mae_multitarget = np.abs(submission_multitarget[target].mean() - sales[target].mean())
    
    mae_comparison.append({
        "Target": target,
        "Train Mean": sales[target].mean(),
        "Baseline MAE": mae_baseline,
        "TargetFit MAE": mae_targetfit,
        "MultiTarget MAE": mae_multitarget,
        "Improvement": f"{(mae_baseline - mae_multitarget) / mae_baseline * 100:.1f}%"
    })

mae_df = pd.DataFrame(mae_comparison)
display(mae_df.round(0))

In [ ]:
## 8. Ensemble Blend - Combine multiple strategies

print("\n" + "=" * 60)
print("ENSEMBLE BLENDING - Combine Baseline, TargetFit, MultiTarget")
print("=" * 60)

# Create weighted ensemble of different approaches
# Weights can be tuned based on validation performance
weights = {
    "baseline": 0.10,      # Light baseline for stability
    "targetfit": 0.3,     # TargetFit works well
    "multitarget": 0.6,   # Multi-target has best variance fit
}

submission_ensemble = baseline_profile.copy()

for target in TARGETS:
    submission_ensemble[target] = (
        weights["baseline"] * baseline_profile[target] +
        weights["targetfit"] * submission_targetfit[target] +
        weights["multitarget"] * submission_multitarget[target]
    )

submission_ensemble[TARGETS] = submission_ensemble[TARGETS].clip(lower=0)

print(f"Weights: {weights}")

# Compare all 4 versions
print("\nAll 4 versions comparison:")
ensemble_comparison = []
for target in TARGETS:
    ensemble_comparison.append({
        "Target": target,
        "Baseline Mean": baseline_profile[target].mean(),
        "Baseline Std": baseline_profile[target].std(),
        "TargetFit Mean": submission_targetfit[target].mean(),
        "TargetFit Std": submission_targetfit[target].std(),
        "MultiTarget Mean": submission_multitarget[target].mean(),
        "MultiTarget Std": submission_multitarget[target].std(),
        "Ensemble Mean": submission_ensemble[target].mean(),
        "Ensemble Std": submission_ensemble[target].std(),
    })

ensemble_df = pd.DataFrame(ensemble_comparison)
display(ensemble_df.round(2))

# Save ensemble submission
ensemble_path = WORK_DIR / "ensemble_submission.csv"
submission_ensemble.to_csv(ensemble_path, index=False)

print(f"\nEnsemble submission saved to: {ensemble_path}")

# Summary of all 4 submissions
print("\n" + "=" * 60)
print("SUBMISSIONS GENERATED:")
print("=" * 60)
print(f"1. baseline_submission.csv        → Simple seasonal baseline")
print(f"2. targetfit_submission.csv       → Target Fit (offset only)")
print(f"3. multitarget_submission.csv     → Multi-Target Scaling (scale + offset)")
print(f"4. ensemble_submission.csv        → Weighted blend of all 3")
print("\nRecommendation: Try multi-target or ensemble for better results!")